# YoloBook

**automated pipeline for training custom YOLO object detection models**

select object classes → download YouTube videos → extract frames → generate labels → train → test

uses the [YouTube Bounding Box](https://research.google.com/youtube-bb/) dataset as a source of labeled video data

In [ ]:
# @title ## STEP 1 : Dataset Parameters

# @markdown Name this new dataset:
dataset = '' # @param {type:'string'}

# @markdown Select the things you want your model to detect:
# @markdown **Animals:**
person = False # @param {type:'boolean'}
cat = False # @param {type:'boolean'}
dog = False # @param {type:'boolean'}
horse = False # @param {type:'boolean'}
cow = False # @param {type:'boolean'}
elephant = False # @param {type:'boolean'}
bear = False # @param {type:'boolean'}
zebra = False # @param {type:'boolean'}
giraffe = False # @param {type:'boolean'}
# @markdown **Vehicles:**
car = False # @param {type:'boolean'}
truck = False # @param {type:'boolean'}
bus = False # @param {type:'boolean'}
boat = False # @param {type:'boolean'}
motorcycle = False # @param {type:'boolean'}
airplane = False # @param {type:'boolean'}
train = False # @param {type:'boolean'}
# @markdown **Other:**
potted_plant = False # @param {type:'boolean'}
toilet = False # @param {type:'boolean'}
skateboard = False # @param {type:'boolean'}
knife = False # @param {type:'boolean'}
bicycle = False # @param {type:'boolean'}
umbrella = False # @param {type:'boolean'}

# @markdown ---
# @markdown **Data split:**
max_videos_to_download = 50 # @param {type:'integer'}
training_percent = 70 # @param {type: 'integer'}
val_test_split = 'Half val set, half test set' # @param ['Use it all for val set', 'Half val set, half test set'] {type: 'string'}
shuffle = True # @param {type:'boolean'}

In [ ]:
# @title ## STEP 2 : Model Selection
# @markdown Choose which size of YOLO11 model to use:
# @markdown *(smaller = faster training, larger = better accuracy)*
network_size = 'small' # @param ['nano', 'small', 'medium', 'large', 'xl'] {type:'string'}

In [ ]:
# @title ## STEP 3 : Environment Setup
# @markdown Colab is auto-detected. For local use, data is stored in `../data/`.

import os
import subprocess

# Auto-detect environment
try:
    from google.colab import drive, files
    use_colab = True
    drive.mount('/content/drive')
    data_base_path = '/content/drive/MyDrive/yolo_data'
    os.makedirs(data_base_path, exist_ok=True)
    print(f"google colab detected — data path: {data_base_path}")
except ImportError:
    use_colab = False
    data_base_path = '../data'
    os.makedirs(data_base_path, exist_ok=True)
    print(f"local environment detected — data path: {data_base_path}")

# Install dependencies
if use_colab:
    subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
    print("dependencies installed")

In [ ]:
# @title ## STEP 4 : Validate Configuration

class_vars = {
    'person': person, 'cat': cat, 'dog': dog, 'horse': horse, 'cow': cow,
    'elephant': elephant, 'bear': bear, 'zebra': zebra, 'giraffe': giraffe,
    'car': car, 'truck': truck, 'bus': bus, 'boat': boat, 'motorcycle': motorcycle,
    'airplane': airplane, 'train': train, 'bicycle': bicycle, 'potted_plant': potted_plant,
    'toilet': toilet, 'skateboard': skateboard, 'knife': knife, 'umbrella': umbrella
}

selected_classes = [name for name, selected in class_vars.items() if selected]

model_map = {'nano': 'yolo11n', 'small': 'yolo11s', 'medium': 'yolo11m', 'large': 'yolo11l', 'xl': 'yolo11x'}

if not dataset.strip():
    print("error: please provide a dataset name in step 1")
elif not selected_classes:
    print("error: please select at least one object class in step 1")
elif max_videos_to_download <= 0:
    print("error: please set max_videos_to_download to a positive number")
else:
    print("configuration validated successfully\n")
    print(f"  dataset:     {dataset}")
    print(f"  classes:     {', '.join(selected_classes)} ({len(selected_classes)} total)")
    print(f"  max videos:  {max_videos_to_download}")
    print(f"  train split: {training_percent}%")
    print(f"  model:       {model_map.get(network_size, 'yolo11s')} ({network_size})")
    print(f"  environment: {'colab' if use_colab else 'local'}")

In [ ]:
# @title ## STEP 5 : Create Dataset Configuration
# @markdown Generates the dataset YAML config and a parameterized processing script.

YTBB_CLASS_MAPPING = {
    'person': 1, 'bicycle': 2, 'car': 3, 'motorcycle': 4, 'airplane': 5,
    'bus': 6, 'train': 7, 'truck': 8, 'boat': 9, 'skateboard': 37,
    'cat': 17, 'dog': 18, 'horse': 19, 'cow': 21, 'elephant': 22,
    'bear': 23, 'zebra': 24, 'giraffe': 25, 'potted_plant': 64,
    'toilet': 70, 'knife': 49, 'umbrella': 28
}

def create_class_remapping(selected_classes):
    ytbb_ids = [YTBB_CLASS_MAPPING[cls] for cls in selected_classes]
    return {ytbb_id: idx for idx, ytbb_id in enumerate(ytbb_ids)}

def create_yaml_config(dataset_name, selected_classes):
    if use_colab:
        data_path = f"{data_base_path}/processed/{dataset_name}/data/"
    else:
        data_path = os.path.abspath(f"../data/processed/{dataset_name}/data/")

    yaml_content = f"""# auto-generated by YoloBook

path: {data_path}
train: images/train
val: images/val
test: images/test

nc: {len(selected_classes)}
names: {selected_classes}
"""

    yaml_path = f"dataset-{dataset_name}.yaml"
    with open(yaml_path, 'w') as f:
        f.write(yaml_content)
    return yaml_path

def modify_process_script():
    with open('src/utility/process-data.py', 'r') as f:
        script = f.read()

    train_ratio = training_percent / 100.0
    remaining = 1.0 - train_ratio
    if val_test_split == 'Half val set, half test set':
        val_ratio = remaining / 2.0
        test_ratio = remaining / 2.0
    else:
        val_ratio = remaining
        test_ratio = 0.0

    class_remapping = create_class_remapping(selected_classes)

    if use_colab:
        raw_data_path = f"{data_base_path}/raw/"
        processed_data_path = f"{data_base_path}/processed/"
    else:
        raw_data_path = "../../data/raw/"
        processed_data_path = "../../data/processed/"

    replacements = {
        "dataset = 'extra'": f"dataset = '{dataset}'",
        "classes = ['dog']": f"classes = {selected_classes}",
        "max_videos_to_download = 50": f"max_videos_to_download = {max_videos_to_download}",
        "class_remapping = {19: 0}": f"class_remapping = {class_remapping}",
        "train_ratio = 0.68": f"train_ratio = {train_ratio}",
        "val_ratio = 0.16": f"val_ratio = {val_ratio}",
        "test_ratio = 0.16": f"test_ratio = {test_ratio}",
        "shuffle = True": f"shuffle = {shuffle}",
        "'../../data/raw/'": f"'{raw_data_path}'",
        "'../../data/processed/'": f"'{processed_data_path}'"
    }

    for old, new in replacements.items():
        script = script.replace(old, new)

    with open('process_data_notebook.py', 'w') as f:
        f.write(script)

yaml_path = create_yaml_config(dataset, selected_classes)
modify_process_script()

print(f"created: {yaml_path}")
print(f"created: process_data_notebook.py")
print(f"\nready to run data processing")

In [ ]:
# @title ## STEP 6 : Run Data Processing
# @markdown Downloads videos, extracts frames, generates labels, and splits data.
# @markdown This will take several hours depending on video count.

start_processing = False # @param {type:'boolean'}

if start_processing:
    print("starting data processing...\n")
    
    raw_dir = f"{data_base_path}/raw"
    processed_dir = f"{data_base_path}/processed"
    os.makedirs(raw_dir, exist_ok=True)
    os.makedirs(processed_dir, exist_ok=True)
    
    try:
        result = subprocess.run(['python', 'process_data_notebook.py'], 
                              capture_output=False, text=True)
        if result.returncode == 0:
            print("\ndata processing completed successfully")
        else:
            print(f"\ndata processing failed with return code: {result.returncode}")
    except Exception as e:
        print(f"error: {e}")
else:
    raw_path = f"{data_base_path}/raw"
    print("set 'start_processing' to True to begin")
    print(f"make sure YTBB csv files are in: {raw_path}")

In [ ]:
# @title ## STEP 7 : Train Model

epochs = 100 # @param {type:'integer'}
batch_size = 16 # @param [8, 16, 32, 64]
start_training = False # @param {type:'boolean'}

if start_training:
    from ultralytics import YOLO

    model_map = {'nano': 'yolo11n', 'small': 'yolo11s', 'medium': 'yolo11m', 'large': 'yolo11l', 'xl': 'yolo11x'}
    model_name = model_map.get(network_size, 'yolo11s')

    print(f"training {model_name} for {epochs} epochs (batch size {batch_size})...\n")

    model = YOLO(f'{model_name}.pt')
    results = model.train(
        data=f'dataset-{dataset}.yaml',
        epochs=epochs,
        batch=batch_size,
        name=f'{dataset}_training'
    )

    print("\ntraining completed")
else:
    model_map = {'nano': 'yolo11n', 'small': 'yolo11s', 'medium': 'yolo11m', 'large': 'yolo11l', 'xl': 'yolo11x'}
    print("set 'start_training' to True to begin")
    print(f"will train {model_map.get(network_size, 'yolo11s')} for {epochs} epochs (batch size {batch_size})")

In [ ]:
# @title ## STEP 8 : Visual Inspection
# @markdown Shows a random image from your dataset with bounding box overlays.

run_visual_check = False # @param {type:'boolean'}
subset = "train" # @param ["train", "val", "test"]

if run_visual_check:
    import cv2
    import matplotlib.pyplot as plt
    import random

    if use_colab:
        images_dir = f"{data_base_path}/processed/{dataset}/data/images/{subset}/"
        labels_dir = f"{data_base_path}/processed/{dataset}/data/labels/{subset}/"
    else:
        images_dir = f"../data/processed/{dataset}/data/images/{subset}/"
        labels_dir = f"../data/processed/{dataset}/data/labels/{subset}/"

    image_files = [f for f in os.listdir(images_dir) if f.endswith('.jpg')]
    chosen = random.choice(image_files)
    name = os.path.splitext(chosen)[0]

    image = cv2.imread(os.path.join(images_dir, chosen))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w, _ = image.shape

    label_path = os.path.join(labels_dir, name + '.txt')
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f:
                _, xc, yc, bw, bh = [float(x) for x in line.split()]
                x1 = int((xc - bw/2) * w)
                y1 = int((yc - bh/2) * h)
                x2 = int((xc + bw/2) * w)
                y2 = int((yc + bh/2) * h)
                cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)

    plt.figure(figsize=(10, 8))
    plt.imshow(image)
    plt.axis('off')
    plt.title(f"{subset}/{chosen}")
    plt.show()
else:
    print("set 'run_visual_check' to True to inspect your dataset")

In [ ]:
# @title ## STEP 9 : Test Your Model
# @markdown Run inference on an image or video using your trained model.

test_type = "image" # @param ["image", "video"]
run_detection = False # @param {type:'boolean'}

if run_detection:
    from ultralytics import YOLO

    model_path = f"runs/detect/{dataset}_training/weights/best.pt"

    if not os.path.exists(model_path):
        print(f"model not found: {model_path}")
        print("please complete training first (step 7)")
    else:
        print(f"running {test_type} detection...\n")

        # Get input file
        filename = None
        if use_colab:
            from google.colab import files
            print(f"upload a {test_type} file:")
            uploaded = files.upload()
            if uploaded:
                filename = list(uploaded.keys())[0]
        else:
            test_input = input(f"enter path to {test_type} file: ").strip()
            if os.path.exists(test_input):
                filename = test_input
            else:
                print(f"file not found: {test_input}")

        if filename:
            model = YOLO(model_path)
            results = model.predict(
                source=filename,
                save=True,
                save_txt=True,
                save_conf=True,
                name=f'{dataset}_detection'
            )

            print(f"\ndetection complete — results saved")

            if use_colab:
                import glob as gl
                output_dir = f"runs/detect/{dataset}_detection"
                for fp in gl.glob(f"{output_dir}/*"):
                    if os.path.isfile(fp):
                        files.download(fp)
else:
    print("set 'run_detection' to True to test your model")
    print(f"expected model: runs/detect/{dataset}_training/weights/best.pt")